# 03 · Limpieza y carga incremental en Silver
1. Deduplica el lote por la **clave de negocio** `(order_id, order_item_id)`.
2. Calcula `hash_fila` (huella del contenido).
3. Aplica `MERGE INTO silver.envios_limpios`:
   * clave nueva → **inserta**
   * clave existente y contenido distinto (y extracción más reciente) → **actualiza**
   * clave existente y mismo contenido (reenvío repetido) → **no escribe nada**
4. Registra en `silver.log_cargas` cuántas filas insertó y actualizó (sale del historial de Delta).

In [0]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from pyspark.sql import functions as F
from src.config import CATALOGO_POR_DEFECTO, T_SILVER, T_STG, T_LOG
from src.transformaciones import deduplicar, anadir_hash_fila
from src.sql_merges import merge_silver
from src.utilidades import crear_tabla_vacia_si_no_existe, merge_con_registro

dbutils.widgets.text("catalogo", CATALOGO_POR_DEFECTO)
dbutils.widgets.text("lote", "lote_01")
catalogo, lote = dbutils.widgets.get("catalogo"), dbutils.widgets.get("lote")
silver = f"{catalogo}.{T_SILVER}"

In [0]:
stg = spark.table(f"{catalogo}.{T_STG}")
assert stg.filter(F.col("lote_id") != lote).count() == 0, "El staging no corresponde a este lote: ejecuta antes 02_validacion con el mismo lote"

filas_validas = stg.count()
limpio = anadir_hash_fila(deduplicar(stg))
filas_tras_dedup = limpio.count()
print(f"Válidas: {filas_validas} | tras deduplicar el lote: {filas_tras_dedup} | duplicados internos eliminados: {filas_validas - filas_tras_dedup}")

## MERGE incremental

In [0]:
crear_tabla_vacia_si_no_existe(spark, silver, limpio)
limpio.createOrReplaceTempView("v_lote_limpio")
filas_before = spark.table(silver).count()

resultado = merge_con_registro(spark, merge_silver(silver, "v_lote_limpio"), catalogo, T_SILVER, lote, T_LOG)
print("Resultado MERGE (lote, tabla, versión, operación, insertadas, actualizadas, borradas):", resultado)

filas_after = spark.table(silver).count()
print(f"Silver: {filas_before} → {filas_after} filas (+{filas_after - filas_before})")
assert filas_after - filas_before == resultado[4], "Las filas nuevas no coinciden con las insertadas por el MERGE"